In [1]:
import pickle
from pathlib import Path
from typing import cast

import numpy as np
import pandas as pd
from commons import (
    DATASET_CLEAN_LOCATION,
    DATASET_CLEAN_UNDERSAMPLING_LOCATION,
    MODEL_FOLDER,
    VECTORIZERS_FOLDER,
    Datasets,
    vectorize_and_split_dataset,
)
from sklearn.metrics import accuracy_score

from language_classifier.custom_components.models.grid_search_cv import GridSearchCVCustom
from language_classifier.custom_components.models.logistic_regression import LogisticRegressionCustom
from language_classifier.custom_components.models.multinomial_nb import MultinomialNBCustom
from language_classifier.custom_components.vectorizers.count_vectorizer import CountVectorizerCustom
from language_classifier.custom_components.vectorizers.tfidf_vectorizer import TfidfVectorizerCustom

# Train Model

Now, we load the datasets that were exported during the data cleaning phase.


In [2]:
df = pd.read_csv(DATASET_CLEAN_LOCATION)
df_undersampling = pd.read_csv(DATASET_CLEAN_UNDERSAMPLING_LOCATION)

I use a utility function from the `commons.py` file, created to avoid duplicating code across notebooks. This function vectorizes the input text using the provided vectorizer and then splits the dataset into training and test sets.

Specifically:

- The function takes a DataFrame containing text and language labels, and a vectorizer (e.g., `CountVectorizerCustom`).
- It transforms the text data into feature vectors with the vectorizer.
- The labels are encoded numerically.
- The dataset is split into training and test subsets with stratification to maintain class distribution.
- The function returns a `Datasets` object that holds the training and test feature matrices, labels, and the original text samples for both sets.

The `Datasets` class is a simple container to keep all these components organized and accessible.



In [3]:
vectorizer_bow = CountVectorizerCustom()
vectorizer_bow_und = CountVectorizerCustom()
vectorizer_tfidf = TfidfVectorizerCustom()
vectorizer_tfidf_und = TfidfVectorizerCustom()

df_bow = vectorize_and_split_dataset(df, vectorizer_bow)
df_bow_undersampling = vectorize_and_split_dataset(df_undersampling, vectorizer_bow_und)
df_tfidf = vectorize_and_split_dataset(df, vectorizer_tfidf)
df_tfidf_undersampling = vectorize_and_split_dataset(df_undersampling, vectorizer_tfidf_und)

I have chosen **Multinomial Naive Bayes (MNB)** and **Logistic Regression (LR)** as our classification algorithms based on insights from the following research articles:

- [Language Identification Using Multinomial Naive Bayes Technique](https://www.researchgate.net/publication/377067809_Language_Identification_Using_Multinomial_Naive_Bayes_Technique)
- [Language Identification Using Combination of Machine Learning Algorithms and Vectorization Techniques](https://www.researchgate.net/publication/362096783_Language_Identification_Using_Combination_of_Machine_Learning_Algorithms_and_Vectorization_Techniques)

**Reason for choosing these models:**

- **Multinomial Naive Bayes:**  
  This model is widely used in text classification tasks due to its simplicity, efficiency, and strong performance, especially when features represent term frequencies. The first article highlights how MNB effectively captures the distribution of words in different languages, making it a natural fit for language identification.

- **Logistic Regression:**  
  Logistic Regression is a robust, interpretable linear model that often performs well on binary and multiclass classification problems. According to the second article, combining logistic regression with appropriate vectorization techniques can improve classification accuracy.

  
For both algorithms I implemented a hyperparameter optimization process using a custom version of GridSearchCV, called **GridSearchCVCustom**. This approach applies **cross-validation**, which is essential for assessing the model’s robustness on unseen data and reducing the risk of overfitting. Specifically, GridSearchCVCustom allows systematic exploration of the hyperparameter space for each algorithm, identifying the combination that maximizes predictive performance based on predefined evaluation metrics.


In [4]:
def naive_bayes(datasets: Datasets) -> GridSearchCVCustom:
    """
    Trains and evaluates a Custom Naive Bayes classifier on the provided dataset.

    This function fits a Multinomial Naive Bayes model using the training data,
    evaluates its accuracy on the test set, and prints the accuracy score as well
    as a summary of misclassified examples including the original text, true label,
    and predicted label.

    Args:
        datasets (Datasets): A Datasets object containing:
            - X (features for training)
            - y (labels for training)
            - X_t (features for testing)
            - y_t (labels for testing)
            - text_t (original text corresponding to test samples)

    Returns:
        MultinomialNBCustom: The trained Custom Naive Bayes classifier.

    """
    nb = GridSearchCVCustom(model_class=MultinomialNBCustom, param_grid={"alpha": [0.001, 0.01, 0.1, 1]})
    nb.fit(datasets.X, datasets.y)
    y_pred = nb.predict(datasets.X_t)
    print("Naive Bayes Custom:", accuracy_score(datasets.y_t, y_pred))

    wrong_idx = datasets.y_t != y_pred
    errors_df = pd.DataFrame({
        "Text": datasets.text_t[wrong_idx],
        "True Label": datasets.y_t[wrong_idx],
        "Predicted Label": y_pred[wrong_idx],
    })
    for _, row in errors_df.iterrows():
        print(f"📝 Text: {row['Text']}")
        print(f"✅ True Label: {row['True Label']}")
        print(f"❌ Predicted: {row['Predicted Label']}")
        print("-" * 50)
    return nb

In [5]:
def logistic_regression(datasets: Datasets) -> GridSearchCVCustom:
    """
    Trains and evaluates a custom logistic regression classifier using grid search with cross-validation.

    This function performs hyperparameter tuning on a logistic regression model using a predefined
    parameter grid and 5-fold cross-validation. It trains the model on the provided training data,
    evaluates accuracy on the test set, and prints the best parameters, cross-validation accuracy,
    and misclassified examples.

    Args:
        datasets (Datasets): A Datasets object containing:
            - X (features for training)
            - y (labels for training)
            - X_t (features for testing)
            - y_t (labels for testing)
            - text_t (original text corresponding to test samples)

    Returns:
        GridSearchCV: The fitted GridSearchCV object containing the best estimator.

    """
    lr = GridSearchCVCustom(model_class=LogisticRegressionCustom, param_grid={"learning_rate": [10, 100, 1000 ], "lambda_coeff": [1e-4, 1e-6]})
    lr.fit(datasets.X, datasets.y)

    y_pred = lr.predict(datasets.X_t)
    print("Logistic Regression:", accuracy_score(datasets.y_t, y_pred))

    wrong_idx = datasets.y_t != y_pred
    errors_df = pd.DataFrame({
        "Text": datasets.text_t[wrong_idx],
        "True Label": datasets.y_t[wrong_idx],
        "Predicted Label": y_pred[wrong_idx],
    })
    for _, row in errors_df.iterrows():
        print(f"📝 Text: {row['Text']}")
        print(f"✅ True Label: {row['True Label']}")
        print(f"❌ Predicted: {row['Predicted Label']}")
        print("-" * 50)

    return lr

We try the Multinomial Naive Bayes and observe that the inputs with undersampling perform better for Bag of Words vectorizations.


In [6]:
nb_bow = naive_bayes(df_bow)

Best params: {'alpha': 0.001}
Naive Bayes Custom: 0.9888727624576681
📝 Text: iletişimde kalın
✅ True Label: 0
❌ Predicted: 1
--------------------------------------------------
📝 Text: ne söylüyordun
✅ True Label: 0
❌ Predicted: 1
--------------------------------------------------
📝 Text: തങങളട ഇടയൽ നഴഞഞകയറയടടണടയകകവനന വകകവരദധര ഭയനന സമഹതതന നരടട തരതതൻ കഴയതത രതയലകക നർദദശചചടടളള തളകളൽ മററ വരതതവൻ വകകപഡയർ കരയനർവവഹകര നയഗചചരകകനന
✅ True Label: 0
❌ Predicted: 1
--------------------------------------------------
📝 Text: afbryder høfligt
✅ True Label: 0
❌ Predicted: 1
--------------------------------------------------
📝 Text: l habituel
✅ True Label: 0
❌ Predicted: 1
--------------------------------------------------
📝 Text: ജനകയ പങകളതതതതലട മതവബസററനകകൾ പരശസത കവരകകൻ വകകപഡയയകക സധചച
✅ True Label: 0
❌ Predicted: 1
--------------------------------------------------
📝 Text: എഴതനന കരയങങൾകകലല തളവകൾ വണ എനനതകണട എലലവരകൾകക ഉറവട ചർതതകളളണ എനനലല
✅ True Label: 0
❌ Predicted: 1
--------------------------------------

In [7]:
nb_bow_und = naive_bayes(df_bow_undersampling)

Best params: {'alpha': 0.001}
Naive Bayes Custom: 0.9963503649635036
📝 Text: progettando
✅ True Label: 1
❌ Predicted: 0
--------------------------------------------------


I display the predicted probabilities from the Naive Bayes model to highlight an important issue: some words (such as "in") appear in multiple languages, which can confuse the classifier and affect its accuracy.

In [8]:
feature_names = vectorizer_bow.get_feature_names_out()
nb_model = nb_bow.best_model
nb_model = cast("MultinomialNBCustom", nb_model)
log_prob = nb_model.log_prob_cond
log_prob = cast("np.ndarray", log_prob)
prob_not_it = np.exp(log_prob[0])
prob_it = np.exp(log_prob[1])
df_prob_nb = pd.DataFrame({
    "word": feature_names,
    "P(word|not it)": prob_not_it,
    "P(word|it)": prob_it,
})

print("Most Important words for not italian class:")
print(df_prob_nb.sort_values("P(word|not it)", ascending=False).head(10))
print("Most Important words for  italian class:")
print(df_prob_nb.sort_values("P(word|it)", ascending=False).head(10))

Most Important words for not italian class:
      word  P(word|not it)    P(word|it)
12396   de        0.018831  7.929102e-08
22       a        0.008927  1.340026e-02
2      the        0.008750  3.172434e-04
10638   en        0.008702  7.929102e-08
8136     क        0.007962  7.929102e-08
12413  que        0.006977  7.929102e-08
3761    la        0.006787  1.554112e-02
8158     ह        0.006080  7.929102e-08
16      of        0.005985  2.379524e-04
1       in        0.005027  1.799914e-02
Most Important words for  italian class:
      word  P(word|not it)  P(word|it)
31740   di    6.793443e-09    0.042659
562      e    2.248636e-03    0.023232
31764  che    6.793443e-09    0.020933
1       in    5.027154e-03    0.017999
31727    è    6.793443e-09    0.016889
15915   un    3.213305e-03    0.015779
3761    la    6.786656e-03    0.015541
15895   il    6.114166e-04    0.014907
22       a    8.926591e-03    0.013400
15906  non    1.290822e-04    0.011973


We also try Logistic Regression and observe that the models trained without undersampling perform better with TF-IDF vectorizations in terms of accuracy. However, this improvement is based solely on accuracy, so other metrics should be carefully analyzed to get a more complete understanding of model performance.

In [9]:
lr_tfidf = logistic_regression(df_tfidf)

Early stopping at epoch 911: dw norm 0.000100 < tol 0.0001
Early stopping at epoch 892: dw norm 0.000100 < tol 0.0001
Early stopping at epoch 899: dw norm 0.000100 < tol 0.0001
Early stopping at epoch 899: dw norm 0.000100 < tol 0.0001
Early stopping at epoch 894: dw norm 0.000100 < tol 0.0001
Early stopping at epoch 884: dw norm 0.000100 < tol 0.0001
Best params: {'learning_rate': 1000, 'lambda_coeff': 1e-06}
Logistic Regression: 0.9941944847605225
📝 Text: e aí
✅ True Label: 0
❌ Predicted: 1
--------------------------------------------------
📝 Text: hai assolutamente ragione
✅ True Label: 1
❌ Predicted: 0
--------------------------------------------------
📝 Text: a dopo
✅ True Label: 1
❌ Predicted: 0
--------------------------------------------------
📝 Text: malheureusement je dois dire non
✅ True Label: 0
❌ Predicted: 1
--------------------------------------------------
📝 Text: non vous avez fait un travail incroyable
✅ True Label: 0
❌ Predicted: 1
-----------------------------------

In [10]:
lr_tfidf_und = logistic_regression(df_tfidf_undersampling)

Best params: {'learning_rate': 100, 'lambda_coeff': 1e-06}
Logistic Regression: 0.9890510948905109
📝 Text: y escuche la pronunciación una o dos veces
✅ True Label: 0
❌ Predicted: 1
--------------------------------------------------
📝 Text: progettando
✅ True Label: 1
❌ Predicted: 0
--------------------------------------------------
📝 Text: à terme il parvint à battre le e meilleur joueur des étatsunis
✅ True Label: 0
❌ Predicted: 1
--------------------------------------------------


After evaluating the performance of both Multinomial Naive Bayes (MNB) and Logistic Regression (LR) classifiers with various vectorization methods and sampling strategies, I observed mixed results. While Logistic Regression trained on the original, imbalanced dataset performed better in accuracy, the undersampled version with MNB and Bag-of-Words (BoW) showed promising results. Therefore, for the next phase of the analysis, I decided to continue analyzing all the 4 combinations to evaluate other metrics



In [11]:
PATH_MODEL_FOLDER = Path(MODEL_FOLDER)
with (PATH_MODEL_FOLDER / "nb_bow.pkl").open("wb") as f:
    pickle.dump(nb_bow, f)

with (PATH_MODEL_FOLDER / "nb_bow_und.pkl").open("wb") as f:
    pickle.dump(nb_bow_und, f)

with (PATH_MODEL_FOLDER / "lr_tfidf.pkl").open("wb") as f:
    pickle.dump(lr_tfidf, f)

with (PATH_MODEL_FOLDER / "lr_tfidf_und.pkl").open("wb") as f:
    pickle.dump(lr_tfidf_und, f)

I save the vectorizers so that I can use them later during the inference phase.


In [12]:
PATH_VECTORIZERS_FOLDER = Path(VECTORIZERS_FOLDER)

with (PATH_VECTORIZERS_FOLDER / "vectorizer_bow.pkl").open("wb") as f:
    pickle.dump(vectorizer_bow , f)

with (PATH_VECTORIZERS_FOLDER / "vectorizer_bow_und.pkl").open("wb") as f:
    pickle.dump(vectorizer_bow_und , f)

with (PATH_VECTORIZERS_FOLDER / "vectorizer_tfidf.pkl").open("wb") as f:
    pickle.dump(vectorizer_tfidf , f)

with (PATH_VECTORIZERS_FOLDER / "vectorizer_tfidf_und.pkl").open("wb") as f:
    pickle.dump(vectorizer_tfidf_und , f)